# 🚨 ข้อมูลเบาะแส/ร้องเรียนยาเสพติด (ปปส.) — ปีงบประมาณ 2560–2569

วิเคราะห์ข้อมูลการ**ตรวจสอบเบาะแสยาเสพติด**รายเดือน แยกตามจังหวัด จากสำนักงาน ป.ป.ส.  
ที่มา: `dataapi.oncb.go.th/suppress/complainVerify` (query จาก API โดยตรง)

- **Treemap** — ภาค ปปส. → จังหวัด (จำนวนเบาะแส)
- **Time Series** — จำนวนเบาะแสรายเดือน + ผลการตรวจสอบ
- **K-Means Clustering** — จัดกลุ่มจังหวัดตามโปรไฟล์การตรวจสอบเบาะแส

In [ ]:
# Google Colab — run this cell first to install dependencies
!pip install pandas plotly scikit-learn statsmodels -q

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.io import to_html
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import os

# monthly verification detail (2560-2569, 77 provinces x 12 months)
df = pd.read_csv('oncb_verify_monthly.csv', encoding='utf-8-sig')
df.columns = df.columns.str.strip()
df['PROV_NAME'] = df['PROV_NAME'].astype(str).str.strip()

# yearly file carries the ปปส. region for each province -> build PROV_ID -> ภาค map
reg = pd.read_csv('oncb_complaint_04.csv', encoding='utf-8-sig')
reg.columns = reg.columns.str.strip()
reg['PROV_ID'] = reg['PROV_ID'].astype(str).str.zfill(2)
# code 10 appears as both 'กทม.' and 'กรุงเทพมหานคร' -> normalise to 'กทม.'
reg['REG_NAME'] = reg['REG_NAME'].replace({'กรุงเทพมหานคร':'กทม.'})
region_map = (reg.sort_values('budget_year')
                 .drop_duplicates('PROV_ID', keep='last')
                 .set_index('PROV_ID')['REG_NAME'].to_dict())

df['PROV_ID'] = df['PROV_ID'].astype(str).str.zfill(2)
df['region']  = df['PROV_ID'].map(region_map).fillna('อื่นๆ')

# monthly datetime (NEWS_YEAR is พ.ศ. -> ค.ศ. = -543)
df['date'] = pd.to_datetime(dict(year=df['NEWS_YEAR']-543,
                                 month=df['NEWS_MONTH_CODE'].astype(int), day=1),
                            errors='coerce')

total_tips = int(df['COMPLAIN_ALL'].sum())
n_prov     = df['PROV_NAME'].nunique()
n_region   = df['region'].nunique()
y_min, y_max = int(df['NEWS_YEAR'].min()), int(df['NEWS_YEAR'].max())
found_rate = df['ALL1'].sum() / max(1, df['ALL123'].sum())

print(f'Rows        : {len(df):,}')
print(f'Tips (total): {total_tips:,}')
print(f'Provinces   : {n_prov}   Regions: {n_region}')
print(f'Years (พ.ศ.) : {y_min} - {y_max}')
print(f'พบพฤติการณ์  : {found_rate:.1%} ของที่ตรวจสอบ')
df.head(3)

In [ ]:
PALETTE = [
    '#003049','#d62828','#f77f00','#fcbf49','#2a9d8f',
    '#264653','#e76f51','#e63946','#457b9d','#1d3557',
    '#06d6a0','#118ab2','#ffd166','#8d99ae','#adb5bd',
]
CLUSTER_COLORS = ['#d62828','#003049','#f77f00','#2a9d8f','#457b9d']

# the 5 target groups (prefix) and their meaning
GROUPS = {
    'FOUNDS'   : 'กลุ่ม1 มีตัวตน·พบพฤติการณ์',
    'NOTFOUNDS': 'กลุ่ม2 มีตัวตน·ไม่พบ',
    'UNKNOW'   : 'กลุ่ม3 ไม่มีตัวตน ทร.14',
    'PLACE'    : 'กลุ่ม4 สถานที่',
    'AREA'     : 'กลุ่ม5 พื้นที่',
}
# the 3 verification results (suffix 1/2/3)
RESULTS = {'ALL1':'พบพฤติการณ์','ALL2':'ไม่พบพฤติการณ์','ALL3':'พิสูจน์ไม่ได้'}

THAI_MONTH = {1:'ม.ค.',2:'ก.พ.',3:'มี.ค.',4:'เม.ย.',5:'พ.ค.',6:'มิ.ย.',
              7:'ก.ค.',8:'ส.ค.',9:'ก.ย.',10:'ต.ค.',11:'พ.ย.',12:'ธ.ค.'}
print('Palette ready.')

## Chart 1 — Treemap: ภาค ปปส. → จังหวัด

ขนาด = จำนวนเบาะแส/ร้องเรียนสะสม (2560–2569) · คลิกเพื่อ zoom

In [ ]:
prov_tot = (df.groupby(['region','PROV_NAME'])['COMPLAIN_ALL'].sum()
              .reset_index().sort_values('COMPLAIN_ALL', ascending=False))
reg_tot  = (df.groupby('region')['COMPLAIN_ALL'].sum()
              .reset_index().sort_values('COMPLAIN_ALL', ascending=False))

ids     = ['ทั้งประเทศ']
labels  = ['ทั้งประเทศ']
parents = ['']
values  = [int(df['COMPLAIN_ALL'].sum())]
mcolors = ['rgba(0,0,0,0)']
cmap={}
for i,(_,r) in enumerate(reg_tot.iterrows()):
    c = PALETTE[i % len(PALETTE)]
    cmap[r['region']] = c
    ids.append(r['region']); labels.append(r['region']); parents.append('ทั้งประเทศ')
    values.append(int(r['COMPLAIN_ALL'])); mcolors.append(c)
for _,r in prov_tot.iterrows():
    ids.append(f"{r['region']}|{r['PROV_NAME']}")
    labels.append(r['PROV_NAME']); parents.append(r['region'])
    values.append(int(r['COMPLAIN_ALL']))
    mcolors.append(cmap.get(r['region'],'#ccc')+'cc')

fig_tree = go.Figure(go.Treemap(
    ids=ids, labels=labels, parents=parents, values=values,
    branchvalues='total',
    marker=dict(colors=mcolors, line=dict(color='white', width=2),
                pad=dict(t=26,l=4,r=4,b=4)),
    texttemplate='<b>%{label}</b><br>%{value:,}',
    textposition='middle center', textfont=dict(size=13,color='white'),
    hovertemplate='<b>%{label}</b><br>เบาะแส: %{value:,}<br>%{percentRoot:.1%} ของทั้งประเทศ<extra></extra>',
    maxdepth=2,
))
fig_tree.update_layout(
    title='เบาะแสยาเสพติด: ภาค ปปส. → จังหวัด (คลิกเพื่อ zoom)',
    height=560, margin=dict(t=50,l=4,r=4,b=4),
    uniformtext=dict(minsize=10, mode='hide'))
fig_tree.show()

## Chart 2 — แผนที่ประเทศไทย: เบาะแสรายจังหวัด

Choropleth · สีเข้ม = เบาะแสมาก · hover ดูอัตราพบพฤติการณ์

In [ ]:
import json, urllib.request
def _load_geojson(local, url):
    try:    return json.load(open(local, encoding='utf-8'))
    except Exception:
        with urllib.request.urlopen(url, timeout=30) as r: return json.load(r)
geojson = _load_geojson('th_provinces.geojson',
    'https://raw.githubusercontent.com/chingchai/OpenGISData-Thailand/master/provinces.geojson')

pmap = df.groupby('PROV_NAME').agg(total=('COMPLAIN_ALL','sum'),
    found=('ALL1','sum'), checked=('ALL123','sum')).reset_index()
pmap['found_rate'] = (pmap['found']/pmap['checked'].replace(0,np.nan)).fillna(0)
fig_map = px.choropleth(pmap, geojson=geojson, featureidkey='properties.pro_th',
    locations='PROV_NAME', color='total', color_continuous_scale='OrRd',
    custom_data=['PROV_NAME','total','found_rate'])
fig_map.update_traces(hovertemplate=('<b>%{customdata[0]}</b><br>เบาะแส: %{customdata[1]:,}<br>'
                                     'พบพฤติการณ์: %{customdata[2]:.1%}<extra></extra>'))
fig_map.update_geos(fitbounds='locations', visible=False)
fig_map.update_layout(title='แผนที่เบาะแสยาเสพติดรายจังหวัด (2560–2569)',
    height=640, margin=dict(t=50,l=0,r=0,b=0), coloraxis_colorbar=dict(title='เบาะแส'))
fig_map.show()

## Chart 3 — Time Series: เบาะแสรายเดือน + ผลการตรวจสอบ

แนวโน้มจำนวนเบาะแสรายเดือน 2560–2569 พร้อมผลการตรวจสอบ 3 แบบ

In [ ]:
monthly = (df.groupby('date')[['COMPLAIN_ALL','ALL1','ALL2','ALL3']]
             .sum().reset_index().sort_values('date'))

fig_ts = go.Figure()
fig_ts.add_trace(go.Scatter(
    x=monthly['date'], y=monthly['COMPLAIN_ALL'],
    mode='lines', name='เบาะแสทั้งหมด',
    line=dict(color='#003049', width=2.8),
    hovertemplate='%{x|%b %Y}<br>เบาะแส: %{y:,}<extra>ทั้งหมด</extra>'))
for col,color in [('ALL1','#2a9d8f'),('ALL2','#f77f00'),('ALL3','#d62828')]:
    fig_ts.add_trace(go.Scatter(
        x=monthly['date'], y=monthly[col], mode='lines', name=RESULTS[col],
        line=dict(color=color, width=1.4, dash='dot'),
        hovertemplate=f'%{{x|%b %Y}}<br>%{{y:,}}<extra>{RESULTS[col]}</extra>'))

fig_ts.update_layout(
    title='จำนวนเบาะแสยาเสพติดรายเดือน (พ.ศ. 2560–2569)',
    xaxis_title='เดือน', yaxis_title='จำนวนเบาะแส (เรื่อง)',
    height=430, plot_bgcolor='white',
    legend=dict(title='<b>รายการ</b>', bgcolor='rgba(255,255,255,.9)',
                bordercolor='#dee2e6', borderwidth=1),
    margin=dict(t=55,b=50,l=65,r=20), hovermode='x unified')
fig_ts.update_xaxes(showgrid=True, gridcolor='#e9ecef', dtick='M12', tickformat='%Y')
fig_ts.update_yaxes(showgrid=True, gridcolor='#e9ecef')
fig_ts.show()

## Chart 4 — Top 15 จังหวัด (จำนวนเบาะแส)

In [ ]:
top_prov = (df.groupby('PROV_NAME')['COMPLAIN_ALL'].sum()
              .sort_values(ascending=False).head(15).reset_index())
top_prov['found'] = top_prov['PROV_NAME'].map(
    df.groupby('PROV_NAME').apply(lambda g: g['ALL1'].sum()/max(1,g['ALL123'].sum())))

fig_bar = go.Figure(go.Bar(
    x=top_prov['COMPLAIN_ALL'], y=top_prov['PROV_NAME'][::-1].values,
    orientation='h',
    marker=dict(color=top_prov['COMPLAIN_ALL'][::-1].values, colorscale='OrRd',
                line=dict(color='white', width=1)),
    text=top_prov['COMPLAIN_ALL'][::-1].apply(lambda v:f'{v:,}'),
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>เบาะแส: %{x:,}<extra></extra>'))
fig_bar.update_layout(
    title='15 จังหวัดที่มีเบาะแสยาเสพติดมากที่สุด (2560–2569)',
    xaxis_title='จำนวนเบาะแส (เรื่อง)', height=470, plot_bgcolor='white',
    margin=dict(t=55,b=50,l=110,r=70))
fig_bar.update_xaxes(showgrid=True, gridcolor='#e9ecef')
fig_bar.show()

## K-Means Clustering — จัดกลุ่มจังหวัดตามโปรไฟล์การตรวจสอบเบาะแส

**Feature ที่ใช้ (12 features)** ต่อจังหวัด:
- **สัดส่วนกลุ่มเป้าหมาย (5)** — % ของเบาะแสที่เป็น กลุ่ม1 มีตัวตน·พบพฤติการณ์ / กลุ่ม2 มีตัวตน·ไม่พบ / กลุ่ม3 ไม่มีตัวตน ทร.14 / กลุ่ม4 สถานที่ / กลุ่ม5 พื้นที่
- **สัดส่วนผลตรวจสอบ (3)** — % พบพฤติการณ์ / ไม่พบ / พิสูจน์ไม่ได้
- **found_rate** — สัดส่วนที่ตรวจสอบแล้วพบพฤติการณ์ (ประสิทธิผล)
- **verify_rate** — สัดส่วนเบาะแสที่ได้รับการตรวจสอบ
- **log_volume** — ปริมาณเบาะแสเฉลี่ยต่อเดือน
- **trend** — แนวโน้มรายปี (เพิ่ม/ลด)

In [ ]:
g = df.groupby('PROV_NAME')
agg = g[['COMPLAIN_ALL','ALL1','ALL2','ALL3','ALL123',
         'FOUNDS','NOTFOUNDS','UNKNOW','PLACE','AREA']].sum()

feat = pd.DataFrame(index=agg.index)
# Features 1-5: target-group proportions
tot = agg['COMPLAIN_ALL'].replace(0,np.nan)
for grp in ['FOUNDS','NOTFOUNDS','UNKNOW','PLACE','AREA']:
    feat['p_'+grp] = (agg[grp]/tot).fillna(0)
# Features 6-8: verification-result proportions
res_tot = agg['ALL123'].replace(0,np.nan)
for r in ['ALL1','ALL2','ALL3']:
    feat['r_'+r] = (agg[r]/res_tot).fillna(0)
# Feature 9: found_rate  10: verify_rate
feat['found_rate']  = (agg['ALL1']/res_tot).fillna(0)
feat['verify_rate'] = (agg['ALL123']/tot).fillna(0)
# Feature 11: log avg monthly volume
n_months = g['date'].nunique()
feat['log_vol'] = np.log1p(agg['COMPLAIN_ALL']/n_months.replace(0,1))
# Feature 12: yearly trend (slope of yearly totals, normalised)
yearly = df.groupby(['PROV_NAME','NEWS_YEAR'])['COMPLAIN_ALL'].sum().reset_index()
def _slope(s):
    x=s['NEWS_YEAR'].values.astype(float); y=s['COMPLAIN_ALL'].values.astype(float)
    if len(x)<2 or x.std()==0: return 0.0
    b=np.polyfit(x,y,1)[0]; return float(b/ (y.mean()+1))
feat['trend'] = yearly.groupby('PROV_NAME').apply(_slope).reindex(feat.index).fillna(0)

# keep provinces with enough data
valid = agg['COMPLAIN_ALL'][agg['COMPLAIN_ALL']>=100].index
feat = feat.loc[feat.index.isin(valid)].copy()
group_cols  = ['p_FOUNDS','p_NOTFOUNDS','p_UNKNOW','p_PLACE','p_AREA']
result_cols = ['r_ALL1','r_ALL2','r_ALL3']

print(f'Feature matrix: {feat.shape[0]} provinces x {feat.shape[1]} features')

X = StandardScaler().fit_transform(feat.values)
k = min(5, max(3, feat.shape[0]//12))
km = KMeans(n_clusters=k, random_state=42, n_init=10)
clab = km.fit_predict(X)
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X); var_exp = pca.explained_variance_ratio_

prov_region = df.drop_duplicates('PROV_NAME').set_index('PROV_NAME')['region'].to_dict()
pca_df = pd.DataFrame({
    'PC1':coords[:,0],'PC2':coords[:,1],'prov':feat.index,
    'cluster':clab.astype(str),
    'total':agg['COMPLAIN_ALL'].reindex(feat.index).values,
    'found_rate':feat['found_rate'].values,
    'verify_rate':feat['verify_rate'].values,
    'region':[prov_region.get(p,'') for p in feat.index],
})
feat_c = feat.copy(); feat_c['cluster']=clab
grp_means = feat_c.groupby('cluster')[group_cols].mean()
meta = feat_c.groupby('cluster')[['found_rate','verify_rate','log_vol','trend']].mean()

print(f'k = {k}   PC1={var_exp[0]:.1%} PC2={var_exp[1]:.1%}')
for c in sorted(pca_df['cluster'].unique()):
    s=pca_df[pca_df['cluster']==c]
    print(f"  กลุ่ม {c}: {len(s):2d} จว.  {int(s['total'].sum()):>7,} เบาะแส  found={s['found_rate'].mean():.1%}")

### Chart 4 — PCA Scatter: K-Means กลุ่มจังหวัด

ขนาดจุด = จำนวนเบาะแส · สี = กลุ่ม

In [ ]:
fig_pca = go.Figure()
for c in sorted(pca_df['cluster'].unique(), key=int):
    sub = pca_df[pca_df['cluster']==c]
    fig_pca.add_trace(go.Scatter(
        x=sub['PC1'], y=sub['PC2'], mode='markers+text', name=f'กลุ่ม {c}',
        marker=dict(size=sub['total'].apply(lambda x: max(12,min(50,np.log1p(x)*4.5))),
                    color=CLUSTER_COLORS[int(c)%len(CLUSTER_COLORS)], opacity=.85,
                    line=dict(color='white', width=1.5)),
        text=sub['prov'], textposition='top center', textfont=dict(size=9),
        customdata=sub[['prov','total','found_rate','verify_rate']].values,
        hovertemplate=('<b>%{customdata[0]}</b><br>เบาะแส: %{customdata[1]:,}<br>'
                       'พบพฤติการณ์: %{customdata[2]:.1%}<br>ตรวจสอบแล้ว: %{customdata[3]:.1%}<extra></extra>')))
fig_pca.update_layout(
    title=(f'K-Means (k={k}) — PCA Projection | 12 features<br>'
           f'<sup>PC1={var_exp[0]:.1%}, PC2={var_exp[1]:.1%} ของความแปรปรวน</sup>'),
    xaxis_title=f'PC1 ({var_exp[0]:.1%})', yaxis_title=f'PC2 ({var_exp[1]:.1%})',
    height=520, plot_bgcolor='white', legend=dict(title='<b>กลุ่ม K-Means</b>'),
    margin=dict(t=75,b=50,l=65,r=20))
fig_pca.update_xaxes(showgrid=True, gridcolor='#e9ecef', zeroline=True, zerolinecolor='#dee2e6')
fig_pca.update_yaxes(showgrid=True, gridcolor='#e9ecef', zeroline=True, zerolinecolor='#dee2e6')
fig_pca.show()

### Chart 5 — Cluster Profiles: แต่ละกลุ่มมีเบาะแสแบบไหน?

In [ ]:
fig_prof = go.Figure()
for i,gcol in enumerate(group_cols):
    nm = GROUPS[gcol.replace('p_','')]
    fig_prof.add_trace(go.Bar(
        name=nm, x=[f'กลุ่ม {c}' for c in grp_means.index], y=grp_means[gcol].values,
        marker_color=PALETTE[i%len(PALETTE)],
        hovertemplate=f'<b>{nm}</b><br>สัดส่วน: %{{y:.1%}}<extra></extra>'))
fig_prof.update_layout(
    barmode='stack',
    title='โปรไฟล์กลุ่มเป้าหมายของแต่ละกลุ่ม K-Means',
    xaxis_title='กลุ่ม K-Means', yaxis_title='สัดส่วนเฉลี่ย',
    yaxis=dict(tickformat='.0%'), height=470, plot_bgcolor='white',
    legend=dict(title='กลุ่มเป้าหมาย', font_size=10, x=1.01, y=.99),
    margin=dict(t=55,b=50,l=70,r=250))
fig_prof.update_yaxes(showgrid=True, gridcolor='#e9ecef')
fig_prof.update_xaxes(showgrid=False)
fig_prof.show()

print('\n── ผลตรวจสอบ & ปริมาณ เฉลี่ยต่อกลุ่ม ──')
print(meta.rename(columns={'found_rate':'พบพฤติการณ์','verify_rate':'ตรวจสอบแล้ว',
                           'log_vol':'log ปริมาณ/เดือน','trend':'แนวโน้ม'}).round(3).to_string())

## 🔮 Predictive Model — พยากรณ์เบาะแสรายเดือน (SARIMA)

**โมเดล:** SARIMA(1,1,1)(1,1,1)₁₂ (time-series forecast)  
แบ่ง train/test โดยกันข้อมูล **12 เดือนสุดท้ายไว้ทดสอบ** → เทียบ *actual vs predicted*  
แล้ว refit ข้อมูลทั้งหมดเพื่อ**พยากรณ์ล่วงหน้า 12 เดือน** พร้อมช่วงความเชื่อมั่น 95%

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from statsmodels.tsa.statespace.sarimax import SARIMAX
ser = df.groupby('date')['COMPLAIN_ALL'].sum().sort_index().asfreq('MS')
H = 12; tr, te = ser.iloc[:-H], ser.iloc[-H:]
m_ho = SARIMAX(tr, order=(1,1,1), seasonal_order=(1,1,1,12),
    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
fc = m_ho.get_forecast(H); pm = fc.predicted_mean
mae  = np.mean(np.abs(pm.values-te.values))
rmse = np.sqrt(np.mean((pm.values-te.values)**2))
mape = np.mean(np.abs((pm.values-te.values)/te.values))*100
# refit on all data -> forecast 12 months ahead
m_full = SARIMAX(ser, order=(1,1,1), seasonal_order=(1,1,1,12),
    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
fut = m_full.get_forecast(12); fm = fut.predicted_mean; fci = fut.conf_int()

fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=ser.index, y=ser.values, mode='lines', name='Actual (ข้อมูลจริง)',
    line=dict(color='#003049', width=2)))
fig_fc.add_trace(go.Scatter(x=te.index, y=pm.values, mode='lines+markers', name='Predicted (ทำนาย·ชุดทดสอบ)',
    line=dict(color='#d62828', width=2.5, dash='dash')))
fig_fc.add_trace(go.Scatter(x=fm.index, y=fm.values, mode='lines+markers', name='Forecast (พยากรณ์ล่วงหน้า 12 เดือน)',
    line=dict(color='#f77f00', width=2.5, dash='dot')))
fig_fc.add_trace(go.Scatter(x=list(fci.index)+list(fci.index[::-1]),
    y=list(fci.iloc[:,1])+list(fci.iloc[:,0][::-1]), fill='toself', fillcolor='rgba(247,127,0,.15)',
    line=dict(color='rgba(0,0,0,0)'), name='ช่วงเชื่อมั่น 95%', hoverinfo='skip'))
fig_fc.update_layout(
    title=f'พยากรณ์เบาะแสรายเดือน (SARIMA) · ทดสอบ 12 เดือน → MAE={mae:.0f}, RMSE={rmse:.0f}, MAPE={mape:.1f}%',
    xaxis_title='เดือน', yaxis_title='จำนวนเบาะแส (เรื่อง)', height=470, plot_bgcolor='white',
    legend=dict(orientation='h', y=-0.22), margin=dict(t=55,b=70,l=65,r=20), hovermode='x unified')
fig_fc.update_xaxes(showgrid=True, gridcolor='#e9ecef')
fig_fc.update_yaxes(showgrid=True, gridcolor='#e9ecef')
fig_fc.show()
print(f'Holdout 12mo → MAE {mae:.0f} | RMSE {rmse:.0f} | MAPE {mape:.1f}%')
print(f'Forecast next-12mo (เฉลี่ย/เดือน): {fm.mean():.0f}')

## Export → pps_viz.html

In [ ]:
CSS = """
*{box-sizing:border-box}
body{font-family:'Sarabun','Segoe UI',sans-serif;background:#eef1f4;margin:0;padding:0;color:#212529}
.page{max-width:1140px;margin:0 auto;padding:32px 24px 64px}
.hero{background:linear-gradient(135deg,#001219 0%,#003049 40%,#d62828 78%,#f77f00 100%);border-radius:16px;padding:40px 48px;margin-bottom:24px;color:white;position:relative;overflow:hidden}
.hero::before{content:'🚨';position:absolute;right:40px;top:12px;font-size:100px;opacity:.12;line-height:1}
.hero h1{margin:0 0 6px;font-size:2.0em;font-weight:700;letter-spacing:-.5px}
.hero p{margin:0;opacity:.75;font-size:1em}
.badge{display:inline-block;background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.25);border-radius:20px;padding:3px 13px;font-size:.83em;margin:12px 6px 0 0}
.stats{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin-bottom:24px}
.stat-card{background:white;border-radius:12px;padding:20px 22px;box-shadow:0 1px 5px rgba(0,0,0,.08);border-left:4px solid var(--ac)}
.stat-card .num{font-size:1.9em;font-weight:700;color:var(--ac);line-height:1.1}
.stat-card .lbl{font-size:.84em;color:#6c757d;margin-top:5px}
.card{background:white;border-radius:12px;box-shadow:0 1px 5px rgba(0,0,0,.08);margin-bottom:24px;overflow:hidden}
.card-header{padding:16px 22px 0;font-size:.78em;font-weight:700;text-transform:uppercase;letter-spacing:.07em;color:#6c757d}
.grid-2{display:grid;grid-template-columns:1fr 1fr;gap:24px}
@media(max-width:768px){.stats{grid-template-columns:1fr 1fr}.grid-2{grid-template-columns:1fr}.hero h1{font-size:1.4em}.hero::before{display:none}}
@keyframes sirenFall{0%{top:-60px;opacity:1}100%{top:110vh;opacity:0}}
@keyframes popIn{0%{opacity:0;transform:translate(-50%,-50%) scale(.05)}60%{transform:translate(-50%,-50%) scale(1.2)}80%{transform:translate(-50%,-50%) scale(.95)}100%{opacity:1;transform:translate(-50%,-50%) scale(1)}}
@keyframes fadeOut{to{opacity:0;transform:translate(-50%,-50%) scale(.6)}}
@keyframes glowPulse{0%,100%{box-shadow:0 0 40px rgba(214,40,40,.7),0 0 80px rgba(247,127,0,.4)}50%{box-shadow:0 0 80px rgba(214,40,40,1),0 0 160px rgba(247,127,0,.7)}}
@keyframes carRun{0%{left:-90px}100%{left:110vw}}
@keyframes redTint{0%{opacity:0}50%{opacity:.16}100%{opacity:0}}
"""

SIREN_JS = r"""<script>
(function(){
  var busy=false;
  function mk(css,html){var e=document.createElement('div');e.style.cssText=css;if(html!==undefined)e.innerHTML=html;return e;}
  function go(){
    if(busy)return; busy=true;
    var trash=[]; function add(e){document.body.appendChild(e);trash.push(e);return e;}
    var ICONS=['🚨','🔍','🚔','🚓','🔦','📋'];
    for(var i=0;i<34;i++){(function(idx){setTimeout(function(){
      add(mk('position:fixed;top:-60px;left:'+(2+idx*2.9)+'%;font-size:'+(16+Math.random()*16)+'px;'+
        'z-index:9980;pointer-events:none;animation:sirenFall '+(1.6+Math.random()*2)+'s linear forwards;opacity:.85',
        ICONS[Math.floor(Math.random()*ICONS.length)]));},idx*60);})(i);}
    var msgs=['🔍 กำลังตรวจสอบเบาะแส 194,795 เรื่อง...','📋 จับคู่ข้อมูลกับ ทร.14...','🚔 ส่งชุดปฏิบัติการลงพื้นที่...',
      '✅ พบพฤติการณ์! ยืนยันเป้าหมาย','🚨 ระดมกวาดล้างทั่วประเทศ','🎯 อัตราการตรวจพบเพิ่มขึ้น','💪 ป.ป.ส. ไม่เคยหยุดพัก'];
    msgs.forEach(function(m,i){setTimeout(function(){
      var t=add(mk('position:fixed;top:'+(8+i*9)+'%;right:24px;z-index:9993;pointer-events:none;'+
        'background:linear-gradient(90deg,#003049,#d62828);color:#fff;font-family:"Sarabun",sans-serif;'+
        'font-size:13px;padding:8px 18px;border-radius:20px;box-shadow:0 2px 12px rgba(214,40,40,.5)',m));
      t.style.transform='translateX(120%)';t.style.transition='transform .4s cubic-bezier(.17,.67,.35,1.3)';
      setTimeout(function(){t.style.transform='translateX(0)';},20);},400+i*350);});
    ['🚔','🚓','🚨','🚓','🚔'].forEach(function(em,i){setTimeout(function(){
      add(mk('position:fixed;bottom:'+(6+i*7)+'%;left:-90px;font-size:'+(30+Math.random()*18)+'px;'+
        'z-index:9985;pointer-events:none;animation:carRun '+(1.4+Math.random()*1.1)+'s linear forwards',em));},2200+i*260);});
    setTimeout(function(){add(mk('position:fixed;inset:0;background:#d62828;z-index:9970;pointer-events:none;animation:redTint 2s ease forwards'));},3200);
    setTimeout(function(){
      var box=add(mk('position:fixed;top:50%;left:50%;z-index:10000;pointer-events:none;text-align:center;'+
        'background:linear-gradient(135deg,#001219 0%,#003049 50%,#d62828 100%);color:white;font-size:26px;font-weight:900;'+
        'padding:32px 52px;border-radius:28px;animation:popIn .65s cubic-bezier(.17,.67,.35,1.3) forwards,glowPulse 1s ease .7s infinite;'+
        'font-family:"Sarabun",sans-serif;line-height:1.7;min-width:360px',
        '🚨 ปฏิบัติการกวาดล้างสำเร็จ! 🚨<br><span style="font-size:17px">ตรวจสอบเบาะแส 194,795 เรื่อง</span><br>'+
        '<span style="font-size:15px;opacity:.85">พื้นที่ปลอดภัยขึ้นแล้ว 🛡️</span><br>'+
        '<span style="font-size:12px;opacity:.6">ขอบคุณทุกเบาะแสจากประชาชน 🙏</span>'));
      setTimeout(function(){box.style.animation='fadeOut .8s ease forwards';
        setTimeout(function(){trash.forEach(function(e){try{e.parentNode.removeChild(e);}catch(x){}});busy=false;},800);},3500);
    },4200);
  }
  window.addEventListener('load',function(){
    var btn=mk('position:fixed;bottom:28px;right:28px;font-size:46px;cursor:pointer;z-index:8000;'+
      'filter:drop-shadow(0 4px 12px rgba(214,40,40,.5));transition:transform .15s;user-select:none;line-height:1','🚨');
    btn.addEventListener('click',go);
    btn.addEventListener('mouseenter',function(){btn.style.transform='scale(1.25) rotate(-10deg)';});
    btn.addEventListener('mouseleave',function(){btn.style.transform='';});
    document.body.appendChild(btn);
    var tip=mk('position:fixed;bottom:84px;right:28px;background:#212529dd;color:white;font-size:12px;padding:5px 11px;'+
      'border-radius:8px;z-index:8000;opacity:0;transition:opacity .2s;pointer-events:none;white-space:nowrap;font-family:Sarabun,sans-serif','กดเพื่อเริ่มปฏิบัติการ! 🚔');
    document.body.appendChild(tip);
    btn.addEventListener('mouseenter',function(){tip.style.opacity='1';});
    btn.addEventListener('mouseleave',function(){tip.style.opacity='0';});
  });
})();
</script>"""

html = f"""<!DOCTYPE html>
<html lang="th"><head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>ปปส. — เบาะแสยาเสพติด 2560-2569</title>
<link href="https://fonts.googleapis.com/css2?family=Sarabun:wght@400;600;700&display=swap" rel="stylesheet">
<style>{CSS}</style>
</head><body><div class="page">
<div class="hero">
  <h1>🚨 เบาะแสยาเสพติด ป.ป.ส. — ปีงบประมาณ 2560–2569</h1>
  <p>วิเคราะห์การตรวจสอบเบาะแส/ร้องเรียนยาเสพติดรายเดือน · Python, Plotly &amp; Scikit-learn</p>
  <span class="badge">สำนักงาน ป.ป.ส. (ONCB)</span>
  <span class="badge">{total_tips:,} เบาะแส</span>
  <span class="badge">{n_prov} จังหวัด</span>
  <span class="badge">{y_min}–{y_max}</span>
  <span class="badge">🔮 SARIMA · MAPE {mape:.1f}%</span>
</div>
<div class="stats">
  <div class="stat-card" style="--ac:#003049"><div class="num">{total_tips:,}</div><div class="lbl">เบาะแสทั้งหมด (เรื่อง)</div></div>
  <div class="stat-card" style="--ac:#d62828"><div class="num">{n_prov}</div><div class="lbl">จังหวัด · {n_region} ภาค</div></div>
  <div class="stat-card" style="--ac:#f77f00"><div class="num">{found_rate:.0%}</div><div class="lbl">ตรวจสอบแล้วพบพฤติการณ์</div></div>
  <div class="stat-card" style="--ac:#2a9d8f"><div class="num">{y_min}–{y_max}</div><div class="lbl">ช่วงปี (พ.ศ.)</div></div>
</div>
<div class="card"><div class="card-header">Treemap — ภาค ปปส. → จังหวัด (คลิกเพื่อ zoom)</div>
  {to_html(fig_tree, full_html=False, include_plotlyjs=True)}</div>
<div class="card"><div class="card-header">🗺️ แผนที่ — เบาะแสรายจังหวัด</div>
  {to_html(fig_map, full_html=False, include_plotlyjs=False)}</div>
<div class="card"><div class="card-header">Time Series — เบาะแสรายเดือน + ผลตรวจสอบ</div>
  {to_html(fig_ts, full_html=False, include_plotlyjs=False)}</div>
<div class="card"><div class="card-header">Top 15 จังหวัด</div>
  {to_html(fig_bar, full_html=False, include_plotlyjs=False)}</div>
<div class="grid-2">
  <div class="card"><div class="card-header">K-Means — PCA Projection</div>
    {to_html(fig_pca, full_html=False, include_plotlyjs=False)}</div>
  <div class="card"><div class="card-header">Cluster Profiles</div>
    {to_html(fig_prof, full_html=False, include_plotlyjs=False)}</div>
</div>
<div class="card"><div class="card-header">🔮 Predictive Model — SARIMA Forecast (Actual vs Predicted)</div>
  {to_html(fig_fc, full_html=False, include_plotlyjs=False)}</div>
</div>
{SIREN_JS}
</body></html>"""

with open('pps_viz.html','w',encoding='utf-8') as f:
    f.write(html)
print(f'Saved: pps_viz.html ({os.path.getsize("pps_viz.html")//1024} KB)')